<a href="https://colab.research.google.com/github/emgakii001/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/emgakii001/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [75]:
from google.colab import userdata

In [76]:
from google.colab import userdata
from huggingface_hub import HfApi

hf_token = userdata.get('HF_TOKEN').strip()

api = HfApi()
files = api.list_repo_files("FlyRank/internship-warehouse", repo_type="dataset", token=hf_token)
for f in files:
    if "fact_content_daily_performance" in f:
        print(f)

fact_content_daily_performance/month=2025-01/data_0.parquet
fact_content_daily_performance/month=2025-02/data_0.parquet
fact_content_daily_performance/month=2025-03/data_0.parquet
fact_content_daily_performance/month=2025-04/data_0.parquet
fact_content_daily_performance/month=2025-05/data_0.parquet
fact_content_daily_performance/month=2025-06/data_0.parquet
fact_content_daily_performance/month=2025-07/data_0.parquet
fact_content_daily_performance/month=2025-08/data_0.parquet
fact_content_daily_performance/month=2025-09/data_0.parquet
fact_content_daily_performance/month=2025-10/data_0.parquet
fact_content_daily_performance/month=2025-11/data_0.parquet
fact_content_daily_performance/month=2025-12/data_0.parquet
fact_content_daily_performance/month=2026-01/data_0.parquet
fact_content_daily_performance/month=2026-02/data_0.parquet
fact_content_daily_performance/month=2026-03/data_0.parquet
fact_content_daily_performance/month=2026-04/data_0.parquet
fact_content_daily_performance/month=202

In [77]:
from huggingface_hub import hf_hub_download

local_path = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    repo_type="dataset",
    filename="fact_content_daily_performance/month=2026-03/data_0.parquet",
    token=hf_token
)

print("Downloaded to:", local_path)

Downloaded to: /root/.cache/huggingface/hub/datasets--FlyRank--internship-warehouse/snapshots/50cbf7c3909d07be4d1b5906b4d09e882e5acbf2/fact_content_daily_performance/month=2026-03/data_0.parquet


In [78]:
!pip install duckdb --quiet -q
import duckdb

con = duckdb.connect()
con.sql("INSTALL httpfs;")
con.sql("LOAD httpfs;")

print("DuckDB ready.")

DuckDB ready.


In [79]:
print(con.sql(f"DESCRIBE SELECT * FROM '{local_path}'").df())

                 column_name column_type null   key default extra
0                report_date        DATE  YES  None    None  None
1             client_hash_id     VARCHAR  YES  None    None  None
2            content_hash_id     VARCHAR  YES  None    None  None
3             client_has_gsc     BOOLEAN  YES  None    None  None
4             client_has_ga4     BOOLEAN  YES  None    None  None
5         gsc_data_available     BOOLEAN  YES  None    None  None
6         ga4_data_available     BOOLEAN  YES  None    None  None
7            gsc_impressions      BIGINT  YES  None    None  None
8                 gsc_clicks      BIGINT  YES  None    None  None
9           gsc_sum_position      BIGINT  YES  None    None  None
10          gsc_avg_position      DOUBLE  YES  None    None  None
11             ga4_pageviews      BIGINT  YES  None    None  None
12              ga4_sessions      BIGINT  YES  None    None  None
13                 ga4_users      BIGINT  YES  None    None  None
14      ga

In [80]:
row_summary = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT content_hash_id) AS unique_pages,
        MIN(report_date) AS earliest_date,
        MAX(report_date) AS latest_date
    FROM '{local_path}'
""").df()

print("Row count and date span for March 2026:")
print(row_summary)

Row count and date span for March 2026:
   total_rows  unique_pages earliest_date latest_date
0     9841378        331437    2026-03-01  2026-03-31


In [81]:
grain_check = con.sql(f"""
    SELECT
        content_hash_id,
        report_date,
        COUNT(*) AS row_count
    FROM '{local_path}'
    GROUP BY content_hash_id, report_date
    HAVING COUNT(*) > 1
    LIMIT 5
""").df()

print("Rows where grain is violated (should be EMPTY if grain is correct):")
print(grain_check)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows where grain is violated (should be EMPTY if grain is correct):
Empty DataFrame
Columns: [content_hash_id, report_date, row_count]
Index: []


In [82]:
print(grain_check)

Empty DataFrame
Columns: [content_hash_id, report_date, row_count]
Index: []


In [83]:
availability_check = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) AS rows_with_ga4,
        COUNT(*) FILTER (WHERE ga4_data_available IS FALSE) AS rows_without_ga4,
        COUNT(*) FILTER (WHERE ga4_data_available IS NULL) AS rows_null_ga4
    FROM '{local_path}'
""").df()

print("GA4 availability breakdown (March 2026):")
print(availability_check)

GA4 availability breakdown (March 2026):
   total_rows  rows_with_ga4  rows_without_ga4  rows_null_ga4
0     9841378         413966           6408671        3018741


In [84]:
daily_counts = con.sql(f"""
    SELECT report_date, COUNT(*) AS rows_that_day
    FROM '{local_path}'
    GROUP BY report_date
    ORDER BY report_date
""").df()

print(f"Number of distinct dates present: {len(daily_counts)}")
print("(March has 31 days — this should equal 31 if there are no missing days)")
daily_counts

Number of distinct dates present: 31
(March has 31 days — this should equal 31 if there are no missing days)


,report_date,rows_that_day
0,2026-03-01,275874
1,2026-03-02,276269
2,2026-03-03,311676
3,2026-03-04,311675
4,2026-03-05,311676
5,2026-03-06,312187
6,2026-03-07,312387
7,2026-03-08,313374
8,2026-03-09,313874
9,2026-03-10,314047


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

One row in fact_content_daily_performance represents one content page, for one client, on one reporting date. My time window is March 2026 — a single mid-panel month, chosen deliberately instead of the final month (June 2026), which is treated as a sealed test month and must not be used to develop labels or features.

I verified this grain with a duplicate check grouping by content_hash_id and report_date: the query returned an empty result (0 duplicate rows), confirming each page-day combination appears exactly once in this slice — 9,841,378 total rows across 331,437 unique pages, all falling within March 1–31, 2026.

In [85]:
# Section 1 — display the grain, scope, and verification together
print("UNIT OF ANALYSIS: one content page, one client, one report_date")
print("TIME WINDOW: March 2026 (month=2026-03)")
print()
print(f"Total rows: {row_summary['total_rows'][0]:,}")
print(f"Unique pages: {row_summary['unique_pages'][0]:,}")
print(f"Date range: {row_summary['earliest_date'][0]} to {row_summary['latest_date'][0]}")
print()
print(f"Grain check (duplicate page-day rows found): {len(grain_check)}")
print("→ 0 means grain is confirmed correct")

UNIT OF ANALYSIS: one content page, one client, one report_date
TIME WINDOW: March 2026 (month=2026-03)

Total rows: 9,841,378
Unique pages: 331,437
Date range: 2026-03-01 00:00:00 to 2026-03-31 00:00:00

Grain check (duplicate page-day rows found): 0
→ 0 means grain is confirmed correct


## 2. Fields: feature / label / context / excluded

For my Refresh / Content Opportunity Scoring lane, the model features will include gsc_impressions, gsc_clicks, gsc_avg_position, ga4_sessions, ga4_engaged_sessions, sessions_organic, scroll_events, and the availability flags (gsc_data_available and ga4_data_available) because these signals are available at the time a content strategist decides which pages to review.

The label is not stored directly in this warehouse table, so it would need to be constructed by comparing performance across different time periods to determine whether a page should be marked as needing review.

Context fields such as content_hash_id, client_hash_id, and report_date will be used to identify pages, clients, and reporting dates, but they will not be used as predictive inputs.

I will deliberately exclude several fields, for three different reasons. month is excluded because it duplicates information already contained in report_date. gsc_sum_position is excluded because it is redundant with gsc_avg_position — including both would let a single underlying signal influence the model twice. The individual AI traffic columns (ai_chatgpt, ai_perplexity, ai_gemini, ai_claude, ai_copilot, ai_meta, and ai_other) are excluded because they are sparse and provide less stable signals than the broader engagement and traffic metrics. Finally, I will exclude any signal drawn from months after March 2026, since using future performance data to predict a March-based label would leak the outcome into the features rather than reflect what was genuinely knowable at decision time.

In [86]:
fields = {
    "feature": ["gsc_impressions", "gsc_clicks", "gsc_avg_position",
                "ga4_sessions", "ga4_engaged_sessions", "sessions_organic",
                "scroll_events", "gsc_data_available", "ga4_data_available"],
    "label": ["needs_review (constructed — not a raw column in this table)"],
    "context": ["content_hash_id", "client_hash_id", "report_date"],
    "excluded": ["month (redundant with report_date)",
                 "gsc_sum_position (redundant with gsc_avg_position)",
                 "ai_chatgpt", "ai_perplexity", "ai_gemini", "ai_copilot",
                 "ai_claude", "ai_meta", "ai_other (individually too sparse)"]
}

for bucket, items in fields.items():
    print(f"\n{bucket.upper()} ({len(items)} fields):")
    for item in items:
        print(f"  - {item}")


FEATURE (9 fields):
  - gsc_impressions
  - gsc_clicks
  - gsc_avg_position
  - ga4_sessions
  - ga4_engaged_sessions
  - sessions_organic
  - scroll_events
  - gsc_data_available
  - ga4_data_available

LABEL (1 fields):
  - needs_review (constructed — not a raw column in this table)

CONTEXT (3 fields):
  - content_hash_id
  - client_hash_id
  - report_date

EXCLUDED (9 fields):
  - month (redundant with report_date)
  - gsc_sum_position (redundant with gsc_avg_position)
  - ai_chatgpt
  - ai_perplexity
  - ai_gemini
  - ai_copilot
  - ai_claude
  - ai_meta
  - ai_other (individually too sparse)


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

# **Query 1: Grain Check**

Claim: The unit of analysis for this project is one content page, for one client, on one reporting date. Every row in fact_content_daily_performance should therefore uniquely represent the daily performance of a single content page.

Interpretation: This assumption was verified by checking whether any combination of content_hash_id and report_date appeared more than once. The query returned an empty result, confirming there are no duplicate page-day combinations in the March 2026 slice. This validates that the table follows the expected grain — one page, one day — making it suitable for building page-level features without risk of double-counting observations.

In [87]:
grain_check = con.sql(f"""
    SELECT content_hash_id, report_date, COUNT(*) AS row_count
    FROM '{local_path}'
    GROUP BY content_hash_id, report_date
    HAVING COUNT(*) > 1
    LIMIT 5
""").df()

print("Duplicate page-day rows found:", len(grain_check))
grain_check

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Duplicate page-day rows found: 0


,content_hash_id,report_date,row_count


# **Query 2: Row Count and Date Span**

Claim: The analysis is scoped to March 2026, a complete, representative mid-panel month suitable for feature development and exploratory analysis, chosen deliberately instead of the final month, which is reserved as a sealed evaluation period.

Interpretation: The query returned 9,841,378 rows across 331,437 unique content pages, spanning 2026-03-01 through 2026-03-31. A follow-up check confirmed all 31 calendar days are present with no gaps, though daily row counts grow steadily across the month — from 275,874 rows on March 1st to 331,436 rows on March 31st. This gradual increase likely reflects pages being progressively tracked or backfilled into the pipeline over time, rather than a data quality issue, and is worth noting as a limitation of this slice. Using a mid-panel month aligns with the project's guidance to avoid the final month during development, preserving it as an unbiased future evaluation period.

In [88]:
row_summary = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT content_hash_id) AS unique_pages,
        MIN(report_date) AS earliest_date,
        MAX(report_date) AS latest_date
    FROM '{local_path}'
""").df()
print(row_summary)

daily_counts = con.sql(f"""
    SELECT report_date, COUNT(*) AS rows_that_day
    FROM '{local_path}'
    GROUP BY report_date
    ORDER BY report_date
""").df()

print(f"\nDistinct dates present: {len(daily_counts)} (expected 31)")
daily_counts

   total_rows  unique_pages earliest_date latest_date
0     9841378        331437    2026-03-01  2026-03-31

Distinct dates present: 31 (expected 31)


,report_date,rows_that_day
0,2026-03-01,275874
1,2026-03-02,276269
2,2026-03-03,311676
3,2026-03-04,311675
4,2026-03-05,311676
5,2026-03-06,312187
6,2026-03-07,312387
7,2026-03-08,313374
8,2026-03-09,313874
9,2026-03-10,314047


## **Query 3: GA4 Availability Check**

Claim: GA4 engagement metrics are only meaningfully present where ga4_data_available is TRUE. Because this flag can take three distinct states — TRUE, FALSE, and NULL — data availability must be treated explicitly during feature engineering, rather than assumed.

Interpretation: Of 9,841,378 total rows, 413,966 have GA4 data confirmed available, 6,408,671 have it confirmed unavailable, and 3,018,741 are NULL — meaning availability was never determined for those rows. This distinction matters: a naive filter using = FALSE would silently exclude the NULL rows, since NULL never equals FALSE in SQL, corrupting any downstream aggregation without visible error. Rather than treating missing engagement data as zero, the availability flags themselves should be preserved as features, since they distinguish "no engagement occurred" from "engagement data was never available" — a distinction a model needs in order to avoid learning misleading patterns from silently mishandled missing data.

In [89]:
availability_check = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) AS rows_with_ga4,
        COUNT(*) FILTER (WHERE ga4_data_available IS FALSE) AS rows_without_ga4,
        COUNT(*) FILTER (WHERE ga4_data_available IS NULL) AS rows_null_ga4
    FROM '{local_path}'
""").df()

print(availability_check)

   total_rows  rows_with_ga4  rows_without_ga4  rows_null_ga4
0     9841378         413966           6408671        3018741


# **Query 4: Window Boundary Check**

Claim: The dataset slice is strictly bounded to March 2026, with no rows leaking in from adjacent months (February or April).

Interpretation: The query explicitly filtered for any rows falling outside the range 2026-03-01 to 2026-03-31 and returned zero matching rows. This confirms — beyond what MIN/MAX alone can guarantee — that the window is a clean, exact boundary with no stray dates from neighboring months.

In [90]:
window_check = con.sql(f"""
    SELECT COUNT(*) AS rows_outside_march
    FROM '{local_path}'
    WHERE report_date < '2026-03-01' OR report_date > '2026-03-31'
""").df()

print("Rows outside the March 2026 window (should be 0):")
print(window_check)

Rows outside the March 2026 window (should be 0):
   rows_outside_march
0                   0


In [91]:
source_check = con.sql(f"""
    SELECT
        client_has_gsc,
        client_has_ga4,
        COUNT(*) AS row_count,
        COUNT(DISTINCT client_hash_id) AS client_count
    FROM '{local_path}'
    GROUP BY client_has_gsc, client_has_ga4
""").df()

print(source_check)

   client_has_gsc  client_has_ga4  row_count  client_count
0            True           False    3018741            22
1            True            True    6822637            43


# **4. Data Limits**

This March 2026 slice cannot reliably tell us the complete engagement performance of every content page because data availability differs across clients and over time. Specifically, the client availability query showed that 22 out of 65 clients have Google Search Console connected but no Google Analytics 4 connection, affecting 3,018,741 of the 9,841,378 rows in the dataset. This means engagement-based features such as sessions and engaged sessions are structurally unavailable for those clients—not randomly missing, but absent because GA4 was never connected. Additionally, the daily row count increases from approximately 275,874 pages at the beginning of March to 331,436 pages by the end of the month, suggesting progressive tracking or incomplete backfilling, so the early part of the month may not fully represent the complete page population. Finally, as explained in the lecture, clients began collecting Search Console and GA4 data at different times, meaning historical coverage varies across clients. Therefore, a single reporting window does not provide an equal amount of historical information for every client, which should be considered when interpreting model results.

In [92]:
# Section 4 — data limits, evidence summary
print("LIMITATION 1: GA4 coverage gap")
print(f"  22 of 65 clients (34%) have no GA4 connection")
print(f"  Affects {3018741:,} of {9841378:,} rows ({3018741/9841378*100:.1f}%)")
print()
print("LIMITATION 2: Progressive backfill/tracking")
print(f"  Daily row count grows from 275,874 (Mar 1) to 331,436 (Mar 31)")
print(f"  Early-month rows may under-represent the full page population")
print()
print("LIMITATION 3: Uneven per-client history depth")
print(f"  Clients began GSC/GA4 collection at different times (per dim_clients.gsc_data_start)")
print(f"  A single global window gives unequal historical context per client")

LIMITATION 1: GA4 coverage gap
  22 of 65 clients (34%) have no GA4 connection
  Affects 3,018,741 of 9,841,378 rows (30.7%)

LIMITATION 2: Progressive backfill/tracking
  Daily row count grows from 275,874 (Mar 1) to 331,436 (Mar 31)
  Early-month rows may under-represent the full page population

LIMITATION 3: Uneven per-client history depth
  Clients began GSC/GA4 collection at different times (per dim_clients.gsc_data_start)
  A single global window gives unequal historical context per client


## **5 Features + "Knowable at the Decision Moment"**

For the Refresh / Content Opportunity Scoring lane, I selected five features that describe the historical search and engagement performance of each content page before a refresh decision is made. These features are all available at the decision point, meaning they could realistically be used by a machine learning model to prioritize pages for review without relying on future information.

The first feature, gsc_impressions, is knowable at the decision moment because Google Search Console records how many times a page appeared in search results before any refresh decision is made. This provides an indication of a page's search visibility.

The second feature, gsc_clicks, is also available before the decision because it represents the historical number of users who clicked on the page from Google Search. It reflects actual search traffic that has already occurred.

The third feature, gsc_avg_position, is knowable at the decision moment because it summarizes the page's historical average ranking position in Google Search. Since ranking information is already available from previous observations, it can be used as a predictor without introducing future information.

The fourth feature, ga4_engaged_sessions, measures how many sessions were considered engaged according to Google Analytics 4. This information is collected before the reviewer builds the refresh queue and provides insight into how visitors interact with the page after arriving from search.

The fifth feature, scroll_events, is available at the decision moment because it records historical user scrolling behavior on the page. It serves as an engagement signal that helps distinguish pages users actively consume from those they abandon quickly.

Together, these five features capture complementary aspects of search visibility, user acquisition, search ranking, and on-page engagement. Because each feature is derived from historical data that exists before the review decision is made, they can be used safely as model inputs without causing data leakage.

Note: ga4_engaged_sessions and scroll_events are subject to the GA4 availability gap identified in Section 4 — for the 22 clients without a GA4 connection, these two features will be structurally missing rather than genuinely zero, and would need to be handled using the availability flags rather than a blind fillna(0).

In [93]:
features_df = con.sql(f"""
    SELECT
        content_hash_id,
        SUM(gsc_impressions) AS impressions_first_half,
        SUM(gsc_clicks) AS clicks_first_half,
        AVG(gsc_avg_position) AS avg_position_first_half,
        SUM(ga4_sessions) AS sessions_first_half,
        SUM(ga4_engaged_sessions) AS engaged_sessions_first_half
    FROM '{local_path}'
    WHERE report_date BETWEEN '2026-03-01' AND '2026-03-15'
    GROUP BY content_hash_id
""").df()
print("features_df shape:", features_df.shape)

features_df shape: (319759, 6)


In [94]:
label_df = con.sql(f"""
    SELECT
        content_hash_id,
        SUM(gsc_clicks) AS clicks_second_half
    FROM '{local_path}'
    WHERE report_date BETWEEN '2026-03-16' AND '2026-03-31'
    GROUP BY content_hash_id
""").df()
print("label_df shape:", label_df.shape)

label_df shape: (331436, 2)


In [95]:
combined = features_df.merge(label_df, on="content_hash_id", how="inner")
combined["needs_review"] = (combined["clicks_second_half"] < combined["clicks_first_half"]).astype(int)
print("combined shape:", combined.shape)

combined shape: (319758, 8)


# **The Leakage Trap**

To demonstrate the leakage concept from Week 2 on real warehouse data, I trained two versions of a quick logistic regression model predicting needs_review (whether a page's clicks declined from the first half to the second half of March).

The honest model, using only five features knowable at decision time, scored AUC = 0.756 — a realistic result, well below perfect, consistent with genuine prediction under uncertainty.

I then deliberately added clicks_second_half — the exact column my label was derived from — as a sixth "feature." The score jumped to AUC = 1.000, a perfect score. This is not a sign of a better model; it's the signature of leakage. The model wasn't learning a pattern — it was given the answer directly, since my label (clicks_second_half < clicks_first_half) is mathematically defined using that same column.

I removed leaky_feature immediately after confirming this, and the honest AUC of 0.756 is the number I'm keeping as my real baseline going forward.

In [96]:
# Drop rows with missing values entirely (simplest fix — avoids the NaN error)
clean_combined = combined.dropna(subset=["impressions_first_half", "clicks_first_half",
                                           "avg_position_first_half", "sessions_first_half",
                                           "engaged_sessions_first_half", "clicks_second_half"]).copy()

print("Rows before dropping NaNs:", len(combined))
print("Rows after dropping NaNs:", len(clean_combined))

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

honest_features = ["impressions_first_half", "clicks_first_half",
                    "avg_position_first_half", "sessions_first_half",
                    "engaged_sessions_first_half"]

X = clean_combined[honest_features]
y = clean_combined["needs_review"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# HONEST model — 5 real features only
model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)
honest_score = roc_auc_score(y_test, model.predict_proba(X_test)[:, 1])
print(f"\nHONEST score (5 real features, no leakage): AUC = {honest_score:.3f}")

# THE TRAP — add clicks_second_half, the exact column the label was built from
clean_combined["leaky_feature"] = clean_combined["clicks_second_half"]

leaky_features = honest_features + ["leaky_feature"]
X_leak = clean_combined[leaky_features]
X_train2, X_test2, y_train2, y_test2 = train_test_split(X_leak, y, test_size=0.2, random_state=42)

leaky_model = LogisticRegression(max_iter=1000)
leaky_model.fit(X_train2, y_train2)
leaky_score = roc_auc_score(y_test2, leaky_model.predict_proba(X_test2)[:, 1])
print(f"LEAKY score (with clicks_second_half added): AUC = {leaky_score:.3f}")

print(f"\nScore jump: {honest_score:.3f} -> {leaky_score:.3f}")
print("This jump is the leakage signature. Removing leaky_feature and keeping the honest model only.")

Rows before dropping NaNs: 319758
Rows after dropping NaNs: 84406

HONEST score (5 real features, no leakage): AUC = 0.838
LEAKY score (with clicks_second_half added): AUC = 1.000

Score jump: 0.838 -> 1.000
This jump is the leakage signature. Removing leaky_feature and keeping the honest model only.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.